# Week 2 Assignment
## Build an end-to-end ML pipeline on sales/price data.

Brief: load data, create monthly features, build pipeline (scaler + model), tune with TimeSeriesSplit, evaluate and save pipeline.


In [4]:
%pip install scikit-learn --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Imports and config
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set(style='whitegrid')

Load the CSV and show a quick head and missing-values check. (I used the dataset file provided.)

In [8]:
df = pd.read_csv('tesla_deliveries_dataset_2015_2025.csv')
print('shape:', df.shape)
display(df.head())
print('Missing values:', df.isnull().sum())

shape: (2640, 12)


,Year,Month,Region,Model,Estimated_Deliveries,Production_Units,Avg_Price_USD,Battery_Capacity_kWh,Range_km,CO2_Saved_tons,Source_Type,Charging_Stations
0,2023,5,Europe,Model S,17646,17922,92874.27,120,704,1863.42,Interpolated (Month),12207
1,2015,2,Asia,Model X,3797,4164,62205.65,75,438,249.46,Official (Quarter),7640
2,2019,1,North America,Model X,8411,9189,117887.32,82,480,605.59,Interpolated (Month),14071
3,2021,2,North America,Model 3,6555,7311,89294.91,120,712,700.07,Official (Quarter),9333
4,2016,12,Middle East,Model Y,12374,13537,114846.78,120,661,1226.88,Estimated (Region),8722


Missing values: Year                    0
Month                   0
Region                  0
Model                   0
Estimated_Deliveries    0
Production_Units        0
Avg_Price_USD           0
Battery_Capacity_kWh    0
Range_km                0
CO2_Saved_tons          0
Source_Type             0
Charging_Stations       0
dtype: int64


Aggregate to monthly totals and create a `date` column.

In [9]:
df['Year'] = df['Year'].astype(int)
df['Month'] = df['Month'].astype(int)
df['date'] = pd.to_datetime(dict(year=df['Year'], month=df['Month'], day=1))
df = df.sort_values('date')
monthly = df.groupby('date', as_index=False).agg({
    'Estimated_Deliveries':'sum',
    'Production_Units':'sum',
    'Avg_Price_USD':'mean'
})
monthly = monthly.reset_index(drop=True)
display(monthly.head())

,date,Estimated_Deliveries,Production_Units,Avg_Price_USD
0,2015-01-01,183180,195793,84502.4970
1,2015-02-01,165053,176119,81745.5695
2,2015-03-01,184567,200151,86221.2895
3,2015-04-01,225623,241706,83446.1640
4,2015-05-01,184264,198205,85632.1110


Create lag features and a short rolling mean to capture autocorrelation.

In [10]:
monthly['lag1'] = monthly['Estimated_Deliveries'].shift(1)
monthly['lag2'] = monthly['Estimated_Deliveries'].shift(2)
monthly['lag3'] = monthly['Estimated_Deliveries'].shift(3)
monthly['rolling_mean_3'] = monthly['Estimated_Deliveries'].shift(1).rolling(3).mean()
monthly = monthly.dropna().reset_index(drop=True)
display(monthly.tail())

,date,Estimated_Deliveries,Production_Units,Avg_Price_USD,lag1,lag2,lag3,rolling_mean_3
124,2025-08-01,214357,225466,85621.9175,201390.0,199951.0,172377.0,191239.333333
125,2025-09-01,193337,205573,88418.6790,214357.0,201390.0,199951.0,205232.666667
126,2025-10-01,178964,190202,83172.4925,193337.0,214357.0,201390.0,203028.000000
127,2025-11-01,197146,213112,85915.3380,178964.0,193337.0,214357.0,195552.666667
128,2025-12-01,209391,225981,78212.7840,197146.0,178964.0,193337.0,189815.666667


Split time-wise: last 12 months as test.

In [11]:
test_months = 12
train = monthly.iloc[:-test_months]
test = monthly.iloc[-test_months:]
FEATURES = ['lag1','lag2','lag3','rolling_mean_3','Production_Units','Avg_Price_USD']
X_train = train[FEATURES]
y_train = train['Estimated_Deliveries']
X_test = test[FEATURES]
y_test = test['Estimated_Deliveries']
print('train:', X_train.shape, 'test:', X_test.shape)

train: (117, 6) test: (12, 6)


Train simple baselines (Linear Regression and RandomForest) for quick comparison.

In [12]:
lr = LinearRegression()
rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
pred_lr_test = lr.predict(X_test)
pred_rf_test = rf.predict(X_test)

In [13]:
def print_metrics(y_true, y_pred, tag=''):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f'{tag} MAE: {mae:.3f}, RMSE: {rmse:.3f}, R2: {r2:.3f}')
print_metrics(y_test, pred_lr_test, 'LR-test')
print_metrics(y_test, pred_rf_test, 'RF-test')

LR-test MAE: 1873.688, RMSE: 2197.105, R2: 0.971
RF-test MAE: 2119.243, RMSE: 2717.493, R2: 0.955


Build a `Pipeline` (scaler + model) and run GridSearchCV with TimeSeriesSplit.

In [14]:
pipe = Pipeline([('scaler', StandardScaler()), ('model', LinearRegression())])
param_grid = [
    {'model': [LinearRegression()]},
    {'model': [RandomForestRegressor(random_state=RANDOM_STATE)], 'model__n_estimators': [50,100], 'model__max_depth':[None,10]}
]
cv = TimeSeriesSplit(n_splits=5)
gs = GridSearchCV(pipe, param_grid, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1, verbose=1)
gs.fit(X_train, y_train)
best_pipeline = gs.best_estimator_
print('Best params:', gs.best_params_)
print('Best CV (neg MAE):', gs.best_score_)

Fitting 5 folds for each of 5 candidates, totalling 25 fits
Best params: {'model': LinearRegression()}
Best CV (neg MAE): -1608.8973392286953


Evaluate the chosen pipeline on the hold-out test set and save a single model package.

In [15]:
pred_test = best_pipeline.predict(X_test)
mae = mean_absolute_error(y_test, pred_test)
rmse = np.sqrt(mean_squared_error(y_test, pred_test))
r2 = r2_score(y_test, pred_test)
model_name = type(best_pipeline.named_steps['model']).__name__.lower()
fname = f'model_package_{model_name}.pkl'
package = {'pipeline': best_pipeline, 'features': FEATURES, 'metrics': {'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2)}}
joblib.dump(package, fname)
print('Saved model package:', fname)
print(f'Test MAE: {mae:.3f}, RMSE: {rmse:.3f}, R2: {r2:.3f}')

Saved model package: model_package_linearregression.pkl
Test MAE: 1873.688, RMSE: 2197.105, R2: 0.971


Loader: safely find the saved package and show short info (use this when submitting to confirm file exists).

In [16]:
import glob, joblib
files = sorted(glob.glob('model_package_*.pkl') + glob.glob('best_pipeline_*.pkl') + glob.glob('best_model_*.pkl'))
if not files:
    print('No model files found in working directory.')
else:
    latest = files[-1]
    obj = joblib.load(latest)
    print('Loaded:', latest)
    if isinstance(obj, dict):
        print('package keys:', list(obj.keys()))
        print('features:', obj.get('features'))
        pipeline = obj.get('pipeline')
    else:
        pipeline = obj
    print('has predict():', hasattr(pipeline, 'predict'))
    if hasattr(pipeline, 'named_steps'):
        print('Pipeline steps:', pipeline.named_steps)

Loaded: model_package_linearregression.pkl
package keys: ['pipeline', 'features', 'metrics']
features: ['lag1', 'lag2', 'lag3', 'rolling_mean_3', 'Production_Units', 'Avg_Price_USD']
has predict(): True
Pipeline steps: {'scaler': StandardScaler(), 'model': LinearRegression()}


Final summary: model name, metrics, and saved filename.

In [17]:
summary = {'model_name': model_name, 'mae': float(mae), 'rmse': float(rmse), 'r2': float(r2), 'saved_file': fname}
print(summary)

{'model_name': 'linearregression', 'mae': 1873.68769379825, 'rmse': 2197.1049407041855, 'r2': 0.9708335083563984, 'saved_file': 'model_package_linearregression.pkl'}
